Import Libraries

In [4]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle

Load Dataset

In [5]:
movies = pd.read_csv("/content/movies.csv")

print("Original Dataset Shape:")
print(movies.shape)

Original Dataset Shape:
(62423, 3)


Reduce Dataset Size
# Start small to avoid RAM crash


In [6]:
movies = movies.head(3000)

print("\nReduced Dataset Shape:")
print(movies.shape)



Reduced Dataset Shape:
(3000, 3)


Check Missing Values

In [7]:
print("\nMissing Values:")
print(movies.isnull().sum())



Missing Values:
movieId    0
title      0
genres     0
dtype: int64


Clean Genres Column

In [8]:
movies["genres"] = movies["genres"].str.replace("|", " ")

print("\nSample Genres:")
print(movies["genres"].head())


Sample Genres:
0    Adventure Animation Children Comedy Fantasy
1                     Adventure Children Fantasy
2                                 Comedy Romance
3                           Comedy Drama Romance
4                                         Comedy
Name: genres, dtype: object


Create Vectorizer

In [9]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

Convert Text To Vectors

In [10]:
vectors = cv.fit_transform(
    movies["genres"]
)

print("\nVector Shape:")
print(vectors.shape)



Vector Shape:
(3000, 21)


Create Similarity Matrix

In [11]:
similarity = cosine_similarity(vectors)

print("\nSimilarity Matrix Shape:")
print(similarity.shape)


Similarity Matrix Shape:
(3000, 3000)


Recommendation Function

In [12]:
def recommend(movie_name):

    # Check movie exists or not
    if movie_name not in movies["title"].values:
        print("Movie not found!")
        return

    # Find movie index
    movie_index = movies[
        movies["title"] == movie_name
    ].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort movies by similarity score
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print("\nRecommended Movies:\n")

    # Print recommended movie titles
    for movie in movie_list:

        print(
            movies.iloc[movie[0]].title
        )


Test Recommendation

In [13]:
recommend("Toy Story (1995)")


Recommended Movies:

Antz (1998)
Kids of the Round Table (1995)
Borrowers, The (1997)
Black Cauldron, The (1985)
Lord of the Rings, The (1978)


In [14]:
recommend("Jumanji (1995)")


Recommended Movies:

Indian in the Cupboard, The (1995)
NeverEnding Story III, The (1994)
Escape to Witch Mountain (1975)
Darby O'Gill and the Little People (1959)
Return to Oz (1985)


In [15]:
recommend("Heat (1995)")


Recommended Movies:

Assassins (1995)
Die Hard: With a Vengeance (1995)
Net, The (1995)
Natural Born Killers (1994)
Judgment Night (1993)


Save Model


In [16]:
pickle.dump(
    movies,
    open("movies.pkl", "wb")
)

pickle.dump(
    similarity,
    open("similarity.pkl", "wb")
)

print("\nModel Saved Successfully!")


Model Saved Successfully!


Evaluation

In [17]:
def genre_accuracy(movie_name):

    movie_index = movies[
        movies["title"] == movie_name
    ].index[0]

    original_genre = set(
        movies.iloc[movie_index]["genres"].split()
    )

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    matched = 0

    for movie in movie_list:

        recommended_genre = set(
            movies.iloc[movie[0]]["genres"].split()
        )

        if len(
            original_genre.intersection(
                recommended_genre
            )
        ) > 0:

            matched += 1

    accuracy = matched / 5

    print("Genre Match Accuracy:", accuracy)

In [18]:
genre_accuracy("Toy Story (1995)")

Genre Match Accuracy: 1.0
